# NCC-PINN - Automated Experiments Runner

This notebook runs automated architecture search experiments for NCC-PINN on Google Colab GPU.

**Features:**
- Mounts Google Drive for persistent storage
- Clones the NCC-PINN repository from GitHub (with branch selection)
- Installs dependencies with CUDA support
- Runs multiple experiments sequentially with different architectures
- Supports Adaptive Expert PINN with domain decomposition
- Auto-generates datasets if missing
- Auto-saves all outputs (plots, metrics, checkpoints) to Drive
- Generates comparison report across all experiments

**Runtime:** GPU (T4 or better recommended)

---


## 1. Check GPU Availability


In [ ]:
# Check if GPU is available
!nvidia-smi


## 2. Mount Google Drive

All outputs will be saved to: `/content/drive/MyDrive/NCC-PINN/`


In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Create output directory on Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NCC-PINN'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f"\n✓ Drive mounted. Outputs will be saved to: {DRIVE_OUTPUT_DIR}")


## 3. Clone Repository & Select Branch

You can select which branch to use (e.g., `main`, `geo-wavelet-domain-decomp`, etc.)


In [ ]:
import subprocess

# Configuration
REPO_URL = "https://github.com/assafzimand/NCC-PINN.git"
REPO_NAME = "NCC-PINN"
WORK_DIR = f"/content/{REPO_NAME}"

# Clone repository if it doesn't exist
if not os.path.exists(WORK_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {WORK_DIR}
else:
    print(f"Repository already exists at {WORK_DIR}")

# Change to repo directory
os.chdir(WORK_DIR)

# Fetch all branches
print("\nFetching latest changes from GitHub...")
!git fetch origin --prune

# Get list of remote branches
result = subprocess.run(['git', 'branch', '-r'], capture_output=True, text=True)
branches = [b.strip().replace('origin/', '') for b in result.stdout.split('\n') 
            if b.strip() and 'HEAD' not in b]

print("\n" + "="*50)
print("Available branches:")
print("="*50)
for i, branch in enumerate(branches, 1):
    marker = " (default)" if branch in ['main', 'master'] else ""
    print(f"  {i}) {branch}{marker}")

# Interactive branch selection
print("\nEnter the number of the branch you want to use:")
branch_choice = input(f"Branch number [1-{len(branches)}]: ").strip()

# Default to first branch if empty
if not branch_choice:
    branch_choice = "1"

try:
    branch_idx = int(branch_choice) - 1
    if 0 <= branch_idx < len(branches):
        selected_branch = branches[branch_idx]
    else:
        print(f"Invalid choice. Using: {branches[0]}")
        selected_branch = branches[0]
except ValueError:
    print(f"Invalid input. Using: {branches[0]}")
    selected_branch = branches[0]

print(f"\n✓ Selected branch: {selected_branch}")

# Checkout and reset to selected branch
print(f"  Switching to branch: {selected_branch}...")
!git checkout -B {selected_branch} origin/{selected_branch}
!git reset --hard origin/{selected_branch}

print(f"\n✓ Working directory: {os.getcwd()}")
print(f"✓ Branch: {selected_branch}")

## 4. Install Dependencies


In [ ]:
# Install PyTorch with CUDA for Colab (CUDA 12.1)
print("Installing PyTorch with CUDA support...")
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --quiet

print("\nInstalling project dependencies...")
%pip install -r requirements.txt --quiet

# Verify CUDA is available
import torch
print(f"\n✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA version: {torch.version.cuda}")
    print(f"✓ GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ WARNING: CUDA not available! Training will be slow on CPU.")


## 5. (Optional) View Experiment Plan

View the current experiment configuration before running.


In [ ]:
# Display current experiment plan
!cat experiments_plan.yaml


## 6. Run Experiments & Auto-Save to Drive

**This cell does everything:**
1. Generates datasets if they don't exist
2. Runs all experiments sequentially
3. For each experiment: trains model, runs analysis (NCC, probes, derivatives, frequency)
4. If Adaptive PINN enabled: spawns expert networks for high-error regions
5. Generates comparison report across all experiments
6. Saves all results (outputs + checkpoints) to Google Drive

**Note:** This may take several hours depending on number of experiments and GPU type.


In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

print("="*70)
print("NCC-PINN: Automated Experiments")
print("="*70)

# Run experiments (unbuffered for real-time output)
!python -u run_experiments.py

print("\n" + "="*70)
print("Experiments Complete! Saving to Google Drive...")
print("="*70)

# Find the experiments directory
experiments_dir = Path("outputs/experiments")
exp_folders = sorted(experiments_dir.glob("*/"), key=lambda x: x.stat().st_mtime)

if exp_folders:
    latest_exp = exp_folders[-1]
    
    # Copy experiment outputs to Drive
    dest = Path(DRIVE_OUTPUT_DIR) / latest_exp.name
    print(f"\nCopying experiment outputs to Drive...")
    shutil.copytree(latest_exp, dest, dirs_exist_ok=True)
    print(f"  ✓ Outputs saved to: {dest}")
    
    # Also copy checkpoints to Drive
    checkpoints_dir = Path("checkpoints")
    if checkpoints_dir.exists():
        checkpoints_dest = dest / "checkpoints"
        print(f"\nCopying checkpoints to Drive...")
        shutil.copytree(checkpoints_dir, checkpoints_dest, dirs_exist_ok=True)
        print(f"  ✓ Checkpoints saved to: {checkpoints_dest}")
    
    print(f"\n" + "="*70)
    print(f"Experiment Directory Structure:")
    print(f"="*70)
    print(f"")
    print(f"{dest.name}/")
    print(f"  ├── experiments_plan.yaml           # Configuration used")
    print(f"  ├── comparison_summary.csv          # Metrics comparison table")
    print(f"  ├── training_and_results_comparison.png")
    print(f"  ├── ncc_classification_comparison.png")
    print(f"  ├── ncc_compactness_comparison.png")
    print(f"  ├── probe_comparison.png")
    print(f"  ├── derivatives_*_comparison.png")
    print(f"  ├── frequency_*_comparison.png")
    print(f"  ├── expert_regions_comparison.png   # (if adaptive PINN enabled)")
    print(f"  │")
    print(f"  ├── <problem>-<arch>-<activation>/  # Per-experiment folder")
    print(f"  │   └── <timestamp>/")
    print(f"  │       ├── config_used.yaml")
    print(f"  │       ├── metrics.json")
    print(f"  │       ├── summary.txt")
    print(f"  │       ├── training_plots/         # Loss curves, predictions")
    print(f"  │       ├── ncc_plots/              # NCC analysis")
    print(f"  │       ├── probe_plots/            # Linear probe analysis")
    print(f"  │       ├── derivatives_plots/      # Derivatives tracking")
    print(f"  │       ├── frequency_plots/        # Frequency analysis")
    print(f"  │       └── adaptive_plots/         # (if adaptive PINN enabled)")
    print(f"  │           ├── expert_regions_epoch_*.png")
    print(f"  │           ├── expert_regions_final.png")
    print(f"  │           └── expert_regions_metadata.json")
    print(f"  │")
    print(f"  └── checkpoints/                    # Model checkpoints")
    print(f"      └── <problem>/")
    print(f"          └── <arch>/")
    print(f"              ├── best_model.pt")
    print(f"              ├── final_model.pt")
    print(f"              └── checkpoint_epoch_*.pt")
    
    print(f"\n" + "="*70)
    print(f"✓ All experiments saved to Google Drive!")
    print(f"="*70)
else:
    print("No experiment folders found. Check for errors above.")


## 7. Notes & Tips

---

### Experiment Configuration
Edit `experiments_plan.yaml` to customize:
- **Base configuration**: problem, epochs, learning rate, optimizer settings
- **Architecture list**: different network architectures to compare
- **Adaptive PINN settings**: enable/disable, max experts, spawn frequency, RF parameters
- **Analysis frequency**: `eval_every`, `inner_metrics_eval_every`

### Adaptive Expert PINN
When `adaptive_pinn.enabled: true`:
- Model starts with a base network covering the entire domain
- Every `spawn_every_epochs`, it uses Random Forest + geometric wavelets to detect high-error regions
- New expert networks are spawned for these regions
- Final solution: `u(x,t) = u_0(x,t) + Σ 1_Ωi(x,t) · u_i(x,t)`
- Overlapping regions are allowed (high-error areas get more neural capacity)

### Key Parameters
```yaml
adaptive_pinn:
  enabled: true/false          # Toggle adaptive PINN
  max_experts: 10              # Maximum expert networks to spawn
  spawn_every_epochs: 1000     # Check for new regions every N epochs
  rf_max_depth: 12             # Deeper = smaller subdomains
  rf_min_samples_leaf: 10      # Smaller = finer regions
  freeze_mode: none/previous/base_only
```

### Tips
- **Use GPU runtime**: Runtime → Change runtime type → GPU (T4 or better)
- **Long runs**: Use Colab Pro for extended sessions (experiments can take hours)
- **Results persist**: All results are auto-saved to Drive with timestamp
- **Monitor progress**: Watch the output logs to track training and expert spawning
- **Compare runs**: Download results from Drive to analyze locally
- **Branch selection**: Use feature branches for experimental features

---
